# 01 · Data access, quality checks and EDA

**Goal:** load the Roboflow pothole dataset reproducibly, add Indian road images from RDD2022, check label quality and understand the data before modelling.

Sources:
- Roboflow public pothole dataset: https://public.roboflow.com/object-detection/pothole (`brad-dwyer/pothole-voxrl`)
- RDD2022 India (class `D40` = pothole): https://github.com/sekilab/RoadDamageDetector (images CC BY-SA 4.0)

In [ ]:
# --- Setup: works locally and on Google Colab (free T4 GPU) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Upload the project folder to Google Drive first (MyDrive/Road-Damage-Pothole-Detection)
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Road-Damage-Pothole-Detection")
    !pip -q install -r requirements.txt
else:
    # run from the notebooks/ folder -> go to the project root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Project root:", os.getcwd())

## 1. Download the Roboflow dataset
Get a free API key at roboflow.com (Settings → API key). Or export a *YOLOv8* ZIP from the website and use `--zip`.

In [ ]:
import os
# os.environ["ROBOFLOW_API_KEY"] = "paste-your-key"   # or set it in your shell, never commit it
!python -m pothole.data.download
# !python -m pothole.data.download --zip "path/to/Pothole.v1.yolov8.zip"

## 2. Add Indian road images (RDD2022 India, ~527 MB)
Only potholes (`D40`) are kept as labels. Some road images without potholes (cracks, normal road) are kept as background examples to reduce false alarms.

In [ ]:
!python -m pothole.data.rdd2022_india
!python -m pothole.data.merge --sources roboflow rdd_india

## 3. Data-quality checks

In [ ]:
import json
from pothole.data.validate import analyse
for name in ["roboflow", "rdd_india", "merged"]:
    try:
        s = analyse(name)
        print(name, json.dumps({k: s[k] for k in ("splits", "issues", "image_sizes_top5", "small_boxes_pct")}, indent=1))
    except SystemExit as e:
        print(name, "skipped:", e)

## 4. EDA plots

In [ ]:
from IPython.display import Image, display
for name in ["roboflow", "rdd_india"]:
    p = f"reports/eda/{name}/eda_overview.png"
    if os.path.exists(p):
        print(name); display(Image(p))

## 5. Look at sample images with their labels

In [ ]:
import random, cv2, matplotlib.pyplot as plt
from pothole.config import PROCESSED_DIR
from pothole.data.yolo_utils import find_split_dirs, list_images, read_labels, label_path_for

def show_samples(dataset, n=6, seed=0):
    imgs = list_images(find_split_dirs(PROCESSED_DIR / dataset)["train"])
    random.Random(seed).shuffle(imgs)
    fig, axes = plt.subplots(2, n // 2, figsize=(16, 7))
    for ax, p in zip(axes.flat, imgs[:n]):
        im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
        for _, xc, yc, bw, bh in read_labels(label_path_for(p)):
            cv2.rectangle(im, (int((xc-bw/2)*w), int((yc-bh/2)*h)), (int((xc+bw/2)*w), int((yc+bh/2)*h)), (255, 0, 0), 3)
        ax.imshow(im); ax.set_title(p.name[:30], fontsize=8); ax.axis("off")
    fig.suptitle(dataset); plt.tight_layout(); plt.show()

show_samples("roboflow")
try: show_samples("rdd_india")
except Exception as e: print("rdd_india not prepared:", e)

## 6. Key observations (fill in)

| Observation | Evidence | Decision |
|---|---|---|
| e.g. many small potholes (<1% of image) | `small_boxes_pct` | keep imgsz 640, check small-object recall |
| e.g. Roboflow images are mostly non-Indian roads | sample grid | add RDD2022 India in the advanced model |
| e.g. duplicate images across splits | `duplicate_across_splits` | remove before training |
| | | |